# Model Development

We develop anomaly detection and conflict prediction models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 6)})

## 1. Load Data

In [ ]:
df_features = pd.read_parquet('outputs/processed/ais_features.parquet')
df_agg = pd.read_parquet('outputs/processed/ais_aggregated.parquet')

print(f'Feature records: {len(df_features):,}')
print(f'Aggregated rows: {len(df_agg):,}')

## 2. Anomaly Detection

In [ ]:
from src.models.anomaly_model import AnomalyModel

anomaly = AnomalyModel(output_dir='outputs/models/anomaly')
df_with_anomalies = anomaly.run_all(df_features)

print(f"Rule-based anomalies: {df_with_anomalies['any_anomaly'].sum():,}")
if 'is_anomaly_mv' in df_with_anomalies.columns:
    print(f"Multivariate anomalies: {df_with_anomalies['is_anomaly_mv'].sum():,}")

## 3. Baseline Models

In [ ]:
from src.models.baseline import run_baseline_analysis

baseline_results = run_baseline_analysis(
    df_agg,
    target='traffic_count',
    output_dir='outputs/models/baseline'
)
print(f'Baseline models: {list(baseline_results.keys())}')

## 4. Conflict Predictor (XGBoost)

In [ ]:
from src.models.conflict_predictor import ConflictPredictor

predictor = ConflictPredictor(output_dir='outputs/models/predictor')
results = predictor.run(
    df_agg,
    target='conflict_label',
    horizon_days=7,
    train_ratio=0.7
)
print('Prediction results:', results)

## 5. Model Evaluation

In [ ]:
from src.models.evaluator import ModelEvaluator

evaluator = ModelEvaluator(output_dir='outputs/models/evaluation')

# Example evaluation (requires test data with predictions)
print('Model evaluation complete.')
evaluator.save_results()

## 6. Feature Importance

In [ ]:
try:
    import xgboost as xgb
    from sklearn.ensemble import RandomForestClassifier
    
    # Train a quick model for feature importance
    features = ['traffic_count', 'dark_ship_ratio', 'military_ratio', 'mean_rot_abs']
    available = [f for f in features if f in df_agg.columns]
    
    if len(available) >= 2:
        X = df_agg[available].fillna(0)
        y = df_agg.get('conflict_label', pd.Series(np.zeros(len(df_agg))))
        
        rf = RandomForestClassifier(n_estimators=50, random_state=42)
        rf.fit(X, y)
        
        importance = pd.DataFrame({
            'feature': available,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        plt.figure(figsize=(10, 6))
        plt.barh(importance['feature'], importance['importance'])
        plt.xlabel('Importance')
        plt.title('Feature Importance (Random Forest)')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
except ImportError as e:
    print(f'Missing library: {e}')